# Hinglish data engine + Whisper LoRA
**Before running:** Runtime → Change runtime type → **T4 GPU**. Key icon (left) → add secrets `SARVAM_API_KEY` and `HF_TOKEN` with *Notebook access* on.

Run cells top to bottom. **Paste each cell's output to Claude before moving on.** Sarvam spend is capped at ₹30 total (`CAP`); finished API calls are cached in `archive/` and never paid for twice.

In [ ]:
# Run this first, and again after any runtime restart. Safe to re-run.
import os, glob, zipfile, subprocess
from google.colab import userdata
os.environ['SARVAM_API_KEY'] = userdata.get('SARVAM_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if not glob.glob('/content/hde_repo/**/hde/pipeline.py', recursive=True):
    zips = glob.glob('/content/**/hinglish_data_engine*.zip', recursive=True)
    if not zips:
        from google.colab import files
        print("Pick D:\\Sarvam\\hinglish_data_engine.zip")
        up = files.upload()
        zips = ['/content/' + list(up)[0]]
    zipfile.ZipFile(zips[0]).extractall('/content/hde_repo')
ROOT = os.path.dirname(os.path.dirname(glob.glob('/content/hde_repo/**/hde/pipeline.py', recursive=True)[0]))
os.chdir(ROOT)
subprocess.run("pip install -q -r requirements.txt", shell=True, check=True)

import torch, torchaudio
from huggingface_hub import HfApi
print("folder:", ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > Change runtime type > T4 GPU")
print("keys loaded:", bool(os.environ['SARVAM_API_KEY']), bool(os.environ['HF_TOKEN']))
print("HF user:", HfApi(token=os.environ['HF_TOKEN']).whoami()['name'])
print("audio-column fix present:", "acol" in open('hde/pipeline.py').read())
CAP = 30   # hard cap on total Sarvam spend in rupees, across all runs

In [ ]:
# Offline tests: no API, no GPU. Must say 50 passed.
%cd {ROOT}
!python -m pytest -q tests

## Smoke test (~₹4)

In [ ]:
# Free: downloads 20 IndicVoices Hindi clips
%cd {ROOT}
!python -m hde.pipeline real --n 20

In [ ]:
# First paid step: 3 synthetic clips (LLM script + Bulbul TTS + Saaras check)
%cd {ROOT}
!python -m hde.pipeline synth --n 3 --max-credits {CAP}

In [ ]:
# Normalise, align (GPU), translate (paid, small), check, validate
%cd {ROOT}
!python -m hde.pipeline finalize --max-credits {CAP}
!python -m hde.pipeline validate
!cat data/stats.json archive/spend.json

In [ ]:
# Look at one record and listen to it
%cd {ROOT}
import json, IPython.display as ipd
rec = json.loads(open('data/manifest.jsonl').readline())
print(json.dumps({k: rec[k] for k in ('id','duration','source','sentences')}, ensure_ascii=False, indent=1))
print(rec['hi_words'][:8])
ipd.Audio(rec['audio'] if os.path.isabs(rec['audio']) else 'data/' + rec['audio'])

In [ ]:
# If manifest is empty or small: why records went to review
%cd {ROOT}
import json, collections
rev = [json.loads(l) for l in open('data/review.jsonl')]
print(len(rev), 'in review')
print(collections.Counter(tuple(r.get('align_flags', [])) + tuple(p[:50] for p in r.get('problems', [])) for r in rev).most_common(8))

## Full run (~₹20 more). Stops cleanly at the cap and keeps everything done.

In [ ]:
%cd {ROOT}
!python -m hde.pipeline real --n 120
!python -m hde.pipeline synth --n 12 --max-credits {CAP}
!python -m hde.pipeline finalize --max-credits {CAP}
!python -m hde.pipeline validate
!cat data/stats.json archive/spend.json

## Whisper-small LoRA: before vs after (~20 min on T4, almost no Sarvam cost)

In [ ]:
%cd {ROOT}
!python -m finetune.whisper_lora --epochs 3 --rank 32 --max-credits {CAP}
!cat runs/whisper_lora/results.json

In [ ]:
# Reproducibility: rebuild from archive only (zero API calls) and compare
%cd {ROOT}
!cp data/manifest.jsonl /content/manifest_first.jsonl
!python -m hde.pipeline finalize --offline
!cmp data/manifest.jsonl /content/manifest_first.jsonl && echo IDENTICAL

In [ ]:
# Save to Drive + zip 5 real clips for the VaaniAgent latency test
%cd {ROOT}
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/sarvam_p1 && cp -r data archive runs /content/drive/MyDrive/sarvam_p1/
!mkdir -p /content/clips && cp $(ls data/audio/rec-r*.wav | head -5) /content/clips/ && cd /content && zip -qr clips.zip clips
from google.colab import files; files.download('/content/clips.zip')